# P5-04 | Bias and Feature-Risk Assessment
**Notebook:** `13b_bias_check.ipynb`  
**Owner:** M2  
**Project:** Hotel Booking Cancellation Prediction

## Objective
Inspect influential features of the team-selected **XGBoost** classifier for **identity/proxy risks, recording artifacts, timing leakage, and deposit-policy feedback**. Test suspected risks with **with-versus-without feature ablations on the original train split**, using five-fold stratified cross-validation.

**No validation/test data used for training or ablation scores.** This notebook is an initial feature-risk audit, **not proof of demographic fairness**. The output requires a human decision and handoff to M4.

**Inputs reviewed:** `13a_model_interpretation.ipynb` (P5-03 interpretation); `10a_model_gradient_boosting.ipynb` (P5-05 preprocessing/model family); **uploaded** `xgb_hotel_cancellation_metadata.json` and `xgb_hotel_cancellation_model.json`.

**Self-contained tuned configuration:** The full, user-provided XGBoost metadata is embedded in Section 5. You **do not have to upload `best_xgb_params.json`, the model JSON, or the fitted train+validation model to Colab**. The provided native model JSON was inspected to verify its tree count, but the previously fitted model is **never used to score or train CV folds**.

## 1. Colab setup and team repository


In [ ]:
from pathlib import Path
import sys, subprocess, importlib, json, warnings

REPO_URL = "https://github.com/MahdhiS/hotel-cancellation.git"
ROOT = Path("/content/hotel-cancellation")

if ROOT.exists() and not (ROOT / ".git").is_dir():
    backup = Path("/content/hotel-cancellation_backup")
    counter = 1
    while backup.exists():
        backup = Path(f"/content/hotel-cancellation_backup_{counter}")
        counter += 1
    ROOT.rename(backup)
    print("Preserved existing non-Git folder at:", backup)

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
else:
    status = subprocess.run(
        ["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"],
        capture_output=True, text=True, check=True
    ).stdout.strip()
    if status:
        print("Existing tracked edits detected. Skipping git pull.")
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", "main"], check=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
TRAIN_PATH = ROOT / "data/processed/shared_prepared/hotel_train_prepared.csv"
OUTPUT_DIR = ROOT / "reports/bias_check"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TRAIN_PATH.is_file():
    raise FileNotFoundError(f"Missing prepared train file: {TRAIN_PATH}. Push P3-05 outputs first.")
print("Repository:", ROOT)
print("Prepared training file:", TRAIN_PATH)
print("Outputs:", OUTPUT_DIR)

Repository: /content/hotel-cancellation
Prepared training file: /content/hotel-cancellation/data/processed/shared_prepared/hotel_train_prepared.csv
Outputs: /content/hotel-cancellation/reports/bias_check


## 2. Load the **training split only**
All feature risk diagnostics and CV experiments use only `hotel_train_prepared.csv`. Validation and test datasets are *not read* in this notebook.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, FunctionTransformer, PowerTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
import sklearn, xgboost

SEED = 42
FOLDS = 5
CV = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

train_prepared = pd.read_csv(TRAIN_PATH)
assert "is_canceled" in train_prepared.columns
assert not train_prepared["is_canceled"].isna().any()
assert set(train_prepared["is_canceled"].unique()) == {0, 1}
assert train_prepared["booking_id"].is_unique

# Country is intentionally not an input to the selected model.
if "country" in train_prepared.columns:
    warnings.warn("country is present in the CSV, but will be excluded from every model experiment.")
y = train_prepared["is_canceled"].astype(int).reset_index(drop=True)
raw_X = train_prepared.drop(columns=["is_canceled", "country"], errors="ignore").reset_index(drop=True)
print("Train rows:", len(train_prepared), "| train predictors before XGBoost engineering:", raw_X.shape[1])
print("Cancellation rate:", round(float(y.mean()), 4))
print("scikit-learn:", sklearn.__version__, "| XGBoost:", xgboost.__version__)
assert len(train_prepared) == len(raw_X) == len(y)
print("Train-only data checks passed. No validation/test CSV was loaded.")

## 3. Review P5-03 interpretation: which features matter?
These importances are **transcribed from the team's `13a_model_interpretation.ipynb`**. They are **validation permutation-importance findings from P5-03**, not newly estimated metrics. **No validation data is loaded here.** Permutation importance may share correlated signal across columns and should not be interpreted as a causal effect.

In [ ]:
importance = pd.DataFrame([
    {"Feature":"agent", "Validation permutation ROC-AUC drop":0.0597, "Std":0.0016},
    {"Feature":"lead_time", "Validation permutation ROC-AUC drop":0.0550, "Std":0.0031},
    {"Feature":"deposit_type", "Validation permutation ROC-AUC drop":0.0233, "Std":0.0009},
    {"Feature":"market_segment", "Validation permutation ROC-AUC drop":0.0217, "Std":0.0018},
    {"Feature":"previous_bookings_not_canceled", "Validation permutation ROC-AUC drop":0.0192, "Std":0.0017},
    {"Feature":"lead_per_night", "Validation permutation ROC-AUC drop":0.0092, "Std":0.0011},
    {"Feature":"adr_ratio", "Validation permutation ROC-AUC drop":0.0035, "Std":0.0007},
    {"Feature":"total_nights", "Validation permutation ROC-AUC drop":0.0029, "Std":0.0008},
])
display(importance)

## 4. Feature-risk screening and decision log
**Identity/proxy risk:** Does a field identify people or correlate with nationality, socioeconomic group, organization or source channel?  
**Recording/timing risk:** Was it known **when the booking was made**? Could the field be a special code, imputed default, or post-booking update?  
**Policy feedback risk:** Is the predictor a decision hotels already made, while the proposed application would recommend that same decision?

Every top-8 feature is reviewed. The flags below are **hypotheses**, not proof of discrimination or leakage. Edit `Decision` and `Evidence / rationale` only after analysing CV and consulting the data owner.

In [ ]:
audit_rows = [
    ("agent", "Possible proxy via agency ID", "Verify agency ID availability at booking; target encoding must be fold-safe", "Test removal; verify encoder fit inside training folds"),
    ("lead_time", "Generally booking behaviour, not identity", "Derived at booking; verify lead-time source and non-negative values", "Usually retain; inspect outliers"),
    ("deposit_type", "Policy-feedback / potential label artifact", "Verify deposit recorded at booking, not changed after outcome; non-refundable cancellations anomalous", "Test removal; policy owner review essential"),
    ("market_segment", "Sales-channel proxy; may correlate with groups", "Confirm the segment is known and consistently coded at booking", "Test removal; document proxy risk"),
    ("previous_bookings_not_canceled", "History-based differential treatment", "Confirm the count contains only history before the current booking", "Verify as-of-time logic; review cold-start impact"),
    ("lead_per_night", "Lead-time proxy, not direct identity", "Derived from lead_time and stay length; check division logic", "Retain if inputs reliable; note correlation"),
    ("adr_ratio", "Price and room-type proxy", "Reference medians must be computed on fold training only", "Verify fold-safe ADR reference and unusual prices"),
    ("total_nights", "Stay behaviour, not direct identity", "Derived from booked nights; verify this is booking-time data", "Usually retain after validation"),
    ("company", "Potential employer identity / corporate proxy", "Verify identifiers are accurate; target encoding requires out-of-fold fitting", "Optional removal test; sensitivity check"),
    ("distribution_channel", "May proxy sales channel/customer segment", "Verify consistent field coding", "Inspect if highly correlated with market_segment"),
    ("country", "Geographic / nationality proxy", "Already excluded from prepared predictors; indirect proxies may remain", "Continue excluding; cannot infer demographic fairness"),
]
audit = pd.DataFrame(audit_rows, columns=["Feature", "Potential risk", "Recording / timing question", "Initial review action"])
audit["Decision"] = "Pending owner review"
audit["Evidence / rationale"] = ""
display(audit)

### 4.1 Training-only recording diagnostics
The following are *descriptive* checks. Raw cancellation rates by agent/deposit/segment are not estimates of protected-group fairness, discrimination, or causation.

In [ ]:
for feature in ("agent", "deposit_type", "market_segment", "company"):
    if feature not in train_prepared.columns:
        print(f"{feature}: not found in prepared train")
        continue
    table = (train_prepared.assign(_cat=train_prepared[feature].fillna("<missing>").astype(str))
             .groupby("_cat", dropna=False)
             .agg(bookings=("is_canceled", "size"), cancellation_rate=("is_canceled", "mean"))
             .sort_values("bookings", ascending=False)
             .head(12))
    print(f"\n{feature}: {train_prepared[feature].nunique(dropna=False)} recorded values")
    display(table)
print("Review abnormal 100% cancellation categories and ambiguous placeholders with M4 / data owner.")

## 5. Exact selected XGBoost configuration from uploaded metadata
The user-provided `xgb_hotel_cancellation_metadata.json` supplies all eight tuned parameters and **206 estimators**. Its values are embedded directly in the next cell so the notebook is runnable in a fresh Colab runtime without manual JSON uploads. The uploaded native `xgb_hotel_cancellation_model.json` contains **206 trees** with **79 model input features**, confirming the saved tree count. A native model JSON does **not** contain the fitted Python preprocessing pipeline or the original training samples.

**CV methodological boundary:** Using P5-05's exact hyperparameters here does not reuse the train+validation-fitted model. Each experiment fits a fresh XGBoost+preprocessing pipeline only on its training fold. The 0.44 decision threshold in metadata is reported for provenance only; **ROC-AUC and PR-AUC are computed using predicted probabilities**, without changing the threshold.

In [ ]:
# Exact settings copied from the two XGBoost exports supplied for this task.
# The metadata is embedded, so no separate JSON upload is needed in Colab.
SOURCE_METADATA = json.loads(r'''
{
  "best_params": {
    "learning_rate": 0.05123681448183182,
    "max_depth": 10,
    "min_child_weight": 13,
    "subsample": 0.7927080266560322,
    "colsample_bytree": 0.47293244725587513,
    "gamma": 4.377980384537018e-07,
    "reg_alpha": 1.9310008498108992e-08,
    "reg_lambda": 8.392507611398146
  },
  "n_estimators": 206,
  "decision_threshold": 0.44,
  "recency_halflife_days": 180,
  "input_columns": [
    "booking_id",
    "hotel",
    "lead_time",
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "meal",
    "market_segment",
    "distribution_channel",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "reserved_room_type",
    "deposit_type",
    "agent",
    "company",
    "adr",
    "total_nights",
    "total_guests",
    "has_kids",
    "season",
    "arrival_day_of_week",
    "arrival_date_month_num",
    "arrival_date",
    "booking_date",
    "days_since_arrival",
    "booking_month",
    "booking_dow",
    "prev_cancel_ratio",
    "adr_per_guest",
    "weekend_share",
    "lead_per_night",
    "adr_ratio"
  ],
  "versions": {
    "sklearn": "1.9.1",
    "xgboost": "3.2.0"
  }
}
''')
CONFIG_SOURCE = "User-provided xgb_hotel_cancellation_metadata.json (embedded)"

assert SOURCE_METADATA["n_estimators"] == 206  # verified against supplied native XGBoost JSON

xgb_params = {**SOURCE_METADATA["best_params"],
              "n_estimators": int(SOURCE_METADATA["n_estimators"])}
xgb_params.update(
    objective="binary:logistic", tree_method="hist", device="cpu",
    random_state=SEED, n_jobs=2
)
RECENCY_HALFLIFE_DAYS = int(SOURCE_METADATA["recency_halflife_days"])
FINAL_MODEL_THRESHOLD = float(SOURCE_METADATA["decision_threshold"])

print("Configuration source:", CONFIG_SOURCE)
print("Number of trees:", xgb_params["n_estimators"])
print("Tree depth:", xgb_params["max_depth"])
print("Learning rate:", xgb_params["learning_rate"])
print("Recency half-life (days):", RECENCY_HALFLIFE_DAYS)
print("Team's final threshold (NOT used to calculate ROC-AUC):", FINAL_MODEL_THRESHOLD)
print("Source scikit-learn:", SOURCE_METADATA["versions"]["sklearn"],
      "| source XGBoost:", SOURCE_METADATA["versions"]["xgboost"])
print("All feature-removal experiments will refit models on training CV folds only.")

## 6. Reproduce booking-time features, with fold-safe references
As in `10a`, construct arrival/booking dates, previous-cancellation ratios, price-per-guest, stay-length ratios, and the hotel/room/month ADR comparison. For each fold, the ADR median is calculated **using only that fold's fitting rows**. Dates for both partitions use a training-fold-based reference; the team's excluded `days_since_arrival` field is present for schema alignment but **never enters the model**.

In [ ]:
def make_arrival_and_booking_dates(df):
    df = df.copy()
    df["agent"] = df["agent"].astype(str)
    df["company"] = df["company"].astype(str)
    df["arrival_date_month_num"] = pd.to_datetime(df["arrival_date_month"], format="%B").dt.month
    df["arrival_date"] = pd.to_datetime(dict(
        year=df["arrival_date_year"].astype(int),
        month=df["arrival_date_month_num"].astype(int),
        day=df["arrival_date_day_of_month"].astype(int),
    ))
    df["booking_date"] = df["arrival_date"] - pd.to_timedelta(df["lead_time"], unit="D")
    return df


def fold_engineering(train_source, holdout_source):
    """Fit price reference from fold training rows, transform both fold partitions."""
    train_source = make_arrival_and_booking_dates(train_source)
    holdout_source = make_arrival_and_booking_dates(holdout_source)
    fold_reference_arrival = train_source["arrival_date"].max()
    ref = (train_source.groupby(["hotel", "reserved_room_type", "arrival_date_month"])["adr"]
           .median().rename("adr_ref"))

    def enrich(df):
        df = df.copy()
        # Included for provenance/input-schema alignment only; excluded from predictors.
        df["days_since_arrival"] = (fold_reference_arrival - df["arrival_date"]).dt.days
        df["booking_month"] = df["booking_date"].dt.month
        df["booking_dow"] = df["booking_date"].dt.dayofweek
        prior = df["previous_cancellations"] + df["previous_bookings_not_canceled"]
        df["prev_cancel_ratio"] = df["previous_cancellations"] / (prior + 1)
        df["adr_per_guest"] = df["adr"] / df["total_guests"].clip(lower=1)
        df["weekend_share"] = df["stays_in_weekend_nights"] / df["total_nights"].clip(lower=1)
        df["lead_per_night"] = df["lead_time"] / df["total_nights"].clip(lower=1)
        df = df.join(ref, on=["hotel", "reserved_room_type", "arrival_date_month"])
        df["adr_ratio"] = df["adr"] / df["adr_ref"].replace(0, np.nan)
        return df.drop(columns=["adr_ref"])

    return enrich(train_source), enrich(holdout_source)


def fold_weights(X_fit, halflife_days=180):
    # The reference date is derived from the fold's own training rows only.
    age = (X_fit["arrival_date"].max() - X_fit["arrival_date"]).dt.days.to_numpy()
    return 0.5 ** (age / halflife_days)

print("Fold-specific booking-time engineering ready.")

## 7. Build the same model family with a removable feature set
Transformers follow `10a`: `TargetEncoder` for `agent`/`company`, one-hot encoding for other categories, square-root transform for lead time, Yeo–Johnson for five numeric columns, passthrough for remaining numeric columns. For ablations, remove a feature **before fitting the preprocessor** and rebuild its selected transformer columns so it cannot re-enter the model. `country`, `booking_id`, and date indicators excluded by the selected recipe are not predictors.

`TargetEncoder.fit_transform()` performs cross-fitted training encodings **inside** each model-fitting CV fold; the held-out CV fold is transformed without seeing its labels.

In [ ]:
TARGET_ENCODE = ["agent", "company"]
SQRT_FEATURES = ["lead_time"]
YJ_FEATURES = ["stays_in_week_nights", "adults", "previous_cancellations", "previous_bookings_not_canceled", "adr"]
DROP_ALWAYS = {"booking_id", "days_since_arrival", "arrival_date_year", "arrival_date", "booking_date", "country", "is_canceled"}


def build_xgb_pipeline(example_X, removed=()):
    removed = set(removed)
    available = set(example_X.columns) - removed - DROP_ALWAYS
    target_features = [c for c in TARGET_ENCODE if c in available]
    onehot_features = [c for c in example_X.select_dtypes(include=["object", "category"]).columns
                       if c in available and c not in target_features]
    sqrt_features = [c for c in SQRT_FEATURES if c in available]
    yj_features = [c for c in YJ_FEATURES if c in available]
    excluded = set(target_features + onehot_features + sqrt_features + yj_features)
    passthrough_features = [c for c in example_X.select_dtypes(include=["number", "bool"]).columns
                            if c in available and c not in excluded]

    transformers = []
    if target_features:
        transformers.append(("target_enc", TargetEncoder(target_type="binary", cv=5, smooth="auto", random_state=SEED), target_features))
    if onehot_features:
        transformers.append(("oh_enc", OneHotEncoder(sparse_output=False, drop=None, handle_unknown="ignore"), onehot_features))
    if sqrt_features:
        transformers.append(("sqrt_transform", FunctionTransformer(
            func=np.sqrt, inverse_func=np.square,
            feature_names_out="one-to-one", validate=True
        ), sqrt_features))
    if yj_features:
        transformers.append(("boxcox_transform", PowerTransformer(method="yeo-johnson", standardize=True), yj_features))
    if passthrough_features:
        transformers.append(("passthrough", "passthrough", passthrough_features))
    if not transformers:
        raise ValueError("No usable predictors remain after ablation.")

    processor = ColumnTransformer(transformers=transformers, remainder="drop", verbose_feature_names_out=True)
    return Pipeline([("preprocessor", processor), ("xgb", XGBClassifier(**xgb_params))])

print("Source-aligned pipeline factory ready.")

## 8. Define the **training CV** experiments
By default, test the three principal potential proxy/policy risks highlighted by `13a`. Each comparison uses the **same five CV folds**, same XGBoost hyperparameters and recency weights. Optional extra tests can be enabled by editing the dictionary. Removing a feature never removes the target or booking ID safeguards.

In [ ]:
ABLATIONS = {
    "Full selected feature set": (),
    "Without agent": ("agent",),
    "Without deposit_type": ("deposit_type",),
    "Without market_segment": ("market_segment",),
    # Optional: uncomment these after the primary check, if M4 requests them.
    # "Without company": ("company",),
    # "Without agent and company": ("agent", "company"),
}

# Preserve identical splits and fold-specific reference prices across experiments.
fold_data = []
for fold_no, (fit_idx, held_idx) in enumerate(CV.split(raw_X, y), start=1):
    X_fit, X_held = fold_engineering(
        raw_X.iloc[fit_idx].copy(), raw_X.iloc[held_idx].copy()
    )
    y_fit = y.iloc[fit_idx].to_numpy()
    y_held = y.iloc[held_idx].to_numpy()
    weights = fold_weights(X_fit, halflife_days=RECENCY_HALFLIFE_DAYS)
    assert "country" not in X_fit.columns and "is_canceled" not in X_fit.columns
    assert len(X_fit) == len(y_fit) == len(weights)
    fold_data.append((fold_no, X_fit, X_held, y_fit, y_held, weights))
print("Prepared", len(fold_data), "identical train-only CV folds for", len(ABLATIONS), "experiments.")
print("Total XGBoost fits to run:", len(fold_data) * len(ABLATIONS))

## 9. Execute the paired ablation experiments
**Note:** This cell can take time on Colab CPU because it trains an XGBoost model for each experiment × CV fold. Do not lower estimator counts or change hyperparameters differently between experiments. The only intended change in each pair is the feature set.

In [ ]:
cv_rows = []
for experiment, removed_features in ABLATIONS.items():
    print(f"\nExperiment: {experiment} | remove={list(removed_features)}")
    for fold_no, X_fit, X_held, y_fit, y_held, weights in fold_data:
        model = build_xgb_pipeline(X_fit, removed=removed_features)
        model.fit(X_fit, y_fit, xgb__sample_weight=weights)
        scores = model.predict_proba(X_held)[:, 1]
        roc_auc = roc_auc_score(y_held, scores)
        pr_auc = average_precision_score(y_held, scores)
        cv_rows.append({
            "Experiment": experiment,
            "Removed features": ", ".join(removed_features) or "None",
            "Fold": fold_no,
            "ROC-AUC": float(roc_auc),
            "PR-AUC": float(pr_auc),
            "Train rows": len(y_fit),
            "CV holdout rows": len(y_held),
        })
        print(f"  Fold {fold_no}: ROC-AUC={roc_auc:.4f}, PR-AUC={pr_auc:.4f}")

cv_details = pd.DataFrame(cv_rows)
display(cv_details)
assert len(cv_details) == FOLDS * len(ABLATIONS)

## 10. Compare paired fold scores
Define `ROC-AUC drop on removal = (full model ROC-AUC) − (without-feature ROC-AUC)`:

- **Positive**: removing this feature reduced ranking performance.
- **Near zero**: little measured effect within CV variation.
- **Negative**: removing this feature improved ranking performance.

The effect on performance is **not itself** a fairness measure and does not dictate the retain/remove decision.

In [ ]:
baseline_scores = (cv_details.loc[cv_details["Experiment"] == "Full selected feature set"]
                   .set_index("Fold")["ROC-AUC"])
summary_rows = []
for name in ABLATIONS:
    exp = cv_details.loc[cv_details["Experiment"] == name].set_index("Fold")
    diffs = baseline_scores - exp["ROC-AUC"]
    summary_rows.append({
        "Experiment": name,
        "Removed features": ", ".join(ABLATIONS[name]) or "None",
        "CV ROC-AUC mean": float(exp["ROC-AUC"].mean()),
        "CV ROC-AUC std": float(exp["ROC-AUC"].std(ddof=1)),
        "CV PR-AUC mean": float(exp["PR-AUC"].mean()),
        "CV PR-AUC std": float(exp["PR-AUC"].std(ddof=1)),
        "ROC-AUC drop on removal": float(diffs.mean()),
        "Paired drop std": float(diffs.std(ddof=1)),
    })
ablation_summary = pd.DataFrame(summary_rows)
display(ablation_summary.round(5))

## 11. Human decisions and handoff
**Required manual step:** After reviewing the ablation scores and confirming booking-time availability/recording rules with your teammate or BA, replace the `Pending owner review` values in `audit["Decision"]` with `Retain`, `Remove`, or `Investigate further`, and record specific evidence in `audit["Evidence / rationale"]`.

If a feature is removed, **notify M4**: the selected preprocessing and the model must be retrained and the model-comparison workflow updated. Because `10a` already displayed test-set results, avoid repeated model-selection cycles driven by that test result. Obtain agreement on the corrected protocol from the team before any further official test use.

`country` is already excluded; this check cannot quantify fairness across nationality groups without an independently approved group-level evaluation protocol.

In [ ]:
# Example only; REMOVE the leading '#' and change after your review:
# audit.loc[audit["Feature"] == "agent", ["Decision", "Evidence / rationale"]] = [
#     "Investigate further", "Agency ID can proxy distribution channel; compare paired train-CV AUC and verify encoding chronology."
# ]

print("Outstanding feature decisions:", int((audit["Decision"] == "Pending owner review").sum()))
display(audit[["Feature", "Potential risk", "Decision", "Evidence / rationale"]])
print("Decision completion requires stakeholder evidence, not only an AUC comparison.")

## 12. Export reproducible evidence for the team
The notebook saves detailed fold scores, paired ablation comparisons, the feature-risk/decision log, P5-03 permutation-importance reference, **the exact model metadata snapshot**, and a short methods statement in `reports/bias_check/`. Download the resulting ZIP for your P5-04 handoff. It deliberately does **not** export a new final fitted classifier: changing features requires M4 to coordinate a separately reviewed retrain.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

cv_details.to_csv(OUTPUT_DIR / "p5_04_cv_ablation_folds.csv", index=False)
ablation_summary.to_csv(OUTPUT_DIR / "p5_04_ablation_summary.csv", index=False)
audit.to_csv(OUTPUT_DIR / "p5_04_feature_risk_decisions.csv", index=False)
importance.to_csv(OUTPUT_DIR / "p5_04_p5_03_importances_reference.csv", index=False)
(OUTPUT_DIR / "p5_04_xgb_configuration.json").write_text(
    json.dumps(SOURCE_METADATA, indent=2), encoding="utf8"
)

notes = (
    "P5-04 bias and feature-risk check\n"
    f"Training data: {TRAIN_PATH} ({len(raw_X)} rows)\n"
    "Validation/test CSVs: NOT LOADED\n"
    f"CV: StratifiedKFold(n_splits={FOLDS}, shuffle=True, random_state={SEED})\n"
    f"XGBoost config source: {CONFIG_SOURCE}\n"
    f"Trees: {xgb_params['n_estimators']}; depth: {xgb_params['max_depth']}; "
    f"recency half-life: {RECENCY_HALFLIFE_DAYS} days\n"
    f"Team final threshold: {FINAL_MODEL_THRESHOLD} (not used for CV ROC-AUC/PR-AUC)\n"
    "Architecture: 10a_model_gradient_boosting.ipynb; importances: 13a_model_interpretation.ipynb\n"
    "ADR price reference computed within each CV fold, using training records only.\n"
    "Hyperparameters held constant across ablations.\n"
    "Ablation results indicate predictive dependence, NOT demographic fairness.\n"
    "Manual signoff required before marking P5-04 complete.\n"
)
(OUTPUT_DIR / "p5_04_methods_and_limits.txt").write_text(notes, encoding="utf8")

ZIP_PATH = OUTPUT_DIR / "13b_bias_check_evidence.zip"
with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT_DIR.glob("p5_04_*")):
        z.write(path, arcname=path.name)
print("Evidence archive:", ZIP_PATH)
print("Completed experiment rows:", len(cv_details))
print("Undecided feature reviews:", int((audit["Decision"] == "Pending owner review").sum()))
print("NO validation or test data loaded; no final model changed.")

# Optional in Google Colab:
# from google.colab import files
# files.download(str(ZIP_PATH))

## 13. Conclusion (complete after running)
Report the actual **train-CV baseline ROC-AUC**, the paired ROC-AUC and PR-AUC changes for removing `agent`, `deposit_type`, and `market_segment`, and whether each risk was verified or remains unresolved. The provided metadata supplies the exact XGBoost settings; **it does not supply ablation results**. Do not copy test/validation metrics into the CV result table.

For each reviewed influential feature, state whether it (a) potentially identifies or proxies guest groups, (b) may reflect post-booking policies or recording quirks, (c) represents reliable booking-time behaviour, or (d) requires stakeholder clarification. Make **Retain / Remove / Investigate further** decisions with evidence, inform M4 of any proposed feature-set change, and post “bias check done” only after signoff.

**Limitations:** Predictive-ablation performance is not a protected-group fairness metric. Country is already excluded. Previously reported P5-05 test performance must not be treated as proof that revised model variants are acceptable.